# MNIST

For this first tutorial we will learn how to build an image classifier for MNIST in PyTorch.

Each MNIST example is basically a 28 × 28 grayscale image representing a digit from 0 to 9.

## Tensors and indexing

We put these pixel values in a tensor with two rows and three columns, so shape gives us (2, 3), and float32 just tells us we’re storing floating-point numbers.

In [ ]:
import torch
from torch import nn

pixels = torch.tensor([[0., 128., 255.],
                       [64., 192., 32.]])
print("Pixels:", pixels)
print("Shape:", pixels.shape)
print("Type:", pixels.dtype)
print("Selected pixel:", pixels[0, 1].item())

**Question:** Change pixels[0, 1] to pixels[1, 0]. Which value do you select?

## Tensor operations

When we divide by 255 it divides every pixel value, so the shape stays the same, and then mean(dim=0) averages down the rows to give one value per column, while mean(dim=1) averages across each row.

In [ ]:
scaled = pixels / 255
print("Scaled pixels:", scaled)
dimension = 0
means = scaled.mean(dim=dimension)
print("Mean:", means)
print("Mean shape:", means.shape)
print("Flattened:", scaled.flatten())

**Question:** Change dimension to 1. Which axis disappears, and how many values remain?

## One weight and its gradient

For this example, prediction = 2 × weight and loss = (prediction − 6)². requires_grad records operations; backward() calculates the derivative.

In [ ]:
weight = torch.tensor(0.0, requires_grad=True)
prediction = 2 * weight
squared_error = (prediction - 6).square()
squared_error.backward()
print("Weight:", weight.item())
print("Prediction:", prediction.item())
print("Squared error:", squared_error.item())
print("Gradient:", weight.grad.item())
with torch.no_grad():
    weight -= 0.1 * weight.grad
print("Updated weight:", weight.item())
print("Updated prediction:", (2 * weight).item())

**Question:** Start with weight 4.0. Which direction does the gradient update move it?

## Data

Load the supplied MNIST sample.

In [ ]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(17)
with np.load("data/mnist-sample.npz") as sample:
    train_images = torch.tensor(sample["train_images"], dtype=torch.float32) / 255
    train_labels = torch.tensor(sample["train_labels"], dtype=torch.long)
    test_images = torch.tensor(sample["test_images"], dtype=torch.float32) / 255
    test_labels = torch.tensor(sample["test_labels"], dtype=torch.long)
image, label = train_images[0], train_labels[0]
print("Training images:", train_images.shape)
print("Test images:", test_images.shape)


### Plotting functions

In [ ]:
def show_image(image):
    plt.figure(figsize=(3, 3))
    plt.imshow(image.detach().cpu(), cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.show()

def show_loss(values):
    plt.plot(range(len(values)), values)
    plt.xlabel("Iteration")
    plt.ylabel("Cross-entropy")
    plt.show()

def show_training(model, inputs, target, values):
    with torch.no_grad():
        probabilities = model(inputs).softmax(dim=-1)
    print("Updates:", len(values) - 1, "target:", target.item(),
          "prediction:", probabilities.argmax().item())
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(range(len(values)), values)
    axes[0].set(xlabel="Update", ylabel="Cross-entropy")
    axes[1].bar(range(10), probabilities.detach().cpu().numpy())
    axes[1].set(xlabel="Digit", ylabel="Probability", xticks=range(10), ylim=(0, 1))
    plt.tight_layout()
    plt.show()


## Images and labels

The pixel values describe the image. The label gives the correct digit.

In [ ]:
show_image(image)
print("Image shape:", image.shape)
print("Label:", label.item())
print("Pixel range:", image.min().item(), image.max().item())

**Question:** Display 1 - image. What happens to the pixel values?

## The input vector

Place the 28 rows end to end to get a vector of 784 pixel values.

In [ ]:
x = image.flatten()
print("Before:", image.shape)
print("After:", x.shape)
print("Number of values:", image.numel(), x.numel())
assert x.shape == (784,)

**Question:** How does each position in the image map to a position in the vector?

## The linear classifier

Linear(784, 10) calculates a score for each digit. Define layers in __init__ and their computation in forward; calling model(x) runs forward.

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(784, 10)

    def forward(self, x):
        return self.linear(x)

torch.manual_seed(17)
model = DigitClassifier()
logits = model(x)
prediction = logits.argmax(dim=-1)
print("Score shape:", logits.shape)
print("Scores:", logits.detach().round(decimals=2))
print("Predicted digit:", prediction.item(), "correct label:", label.item())

## Cross-entropy

Cross-entropy compares the scores with the correct label.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
logits = model(x)
prediction = logits.argmax(dim=-1)
loss = loss_fn(logits, label)
print("Loss:", round(loss.item(), 4))
print("Loss shape:", loss.shape)

**Question:** Use prediction as the first argument to loss_fn. What does PyTorch require there?

## Updating the parameters

backward() calculates gradients. step() updates the parameters.

In [ ]:
torch.manual_seed(17)
model = DigitClassifier()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
before = model.linear.weight.detach().clone()
optimizer.zero_grad()
loss = loss_fn(model(x), label)
loss.backward()
print("Gradient norm:", round(model.linear.weight.grad.norm().item(), 4))
print("Weights changed after backward:", not torch.equal(before, model.linear.weight))
optimizer.step()
print("Weights changed after step:", not torch.equal(before, model.linear.weight))

**Question:** Comment out optimizer.step(). Which values change?

## Training on one image

Repeat the update and inspect the loss and predicted digit.

In [ ]:
torch.manual_seed(17)
memoriser = DigitClassifier()
optimizer = torch.optim.SGD(memoriser.parameters(), lr=0.1)
target = label
losses = []
for update in range(100):
    optimizer.zero_grad()
    loss = loss_fn(memoriser(x), target)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print("Training target:", target.item(), "actual label:", label.item())
print("Predicted digit:", memoriser(x).argmax(dim=-1).item())
print("Loss:", round(losses[0], 4), "→", round(losses[-1], 4))
show_loss(losses)

**Question:** Set target to torch.tensor(3). Which digit does the model learn to predict?

## Training updates

Step through training and compare the loss with the probability assigned to each digit.

In [ ]:
torch.manual_seed(17)
live_model = DigitClassifier()
live_optimizer = torch.optim.SGD(live_model.parameters(), lr=0.03)
live_target = label.clone()
live_losses = [loss_fn(live_model(x), live_target).item()]

def train_steps(steps=1):
    for _ in range(steps):
        live_optimizer.zero_grad()
        loss = loss_fn(live_model(x), live_target)
        loss.backward()
        live_optimizer.step()
        with torch.no_grad():
            live_losses.append(loss_fn(live_model(x), live_target).item())
    show_training(live_model, x, live_target, live_losses)

train_steps(0)

In [ ]:
train_steps(10)

**Question:** Train against target 3, then reset and train against target 5. What does a low loss establish?

## Gradient accumulation

Each call to backward() adds to the stored gradients.

In [ ]:
torch.manual_seed(17)
probe = DigitClassifier()
probe.zero_grad()
loss_fn(probe(x), label).backward()
first_gradient = probe.linear.weight.grad.clone()
probe.zero_grad()  # What if we remove just this line?
loss_fn(probe(x), label).backward()
second_gradient = probe.linear.weight.grad.clone()
print("First norm:", round(first_gradient.norm().item(), 4))
print("Second / first:", round((second_gradient.norm() / first_gradient.norm()).item(), 4))

**Question:** Remove the second probe.zero_grad(). How does the gradient change?

## Batches

Process several images in one tensor operation.

In [ ]:
def train_small(batch_size, epochs=5, model_class=DigitClassifier):
    torch.manual_seed(17)
    trained = model_class()
    optimizer = torch.optim.SGD(trained.parameters(), lr=0.1)
    steps = 0
    started = time.perf_counter()
    for epoch in range(epochs):
        for start in range(0, len(train_images), batch_size):
            inputs = train_images[start:start + batch_size].flatten(start_dim=1)
            targets = train_labels[start:start + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(trained(inputs), targets)
            loss.backward()
            optimizer.step()
            steps += 1
    seconds = time.perf_counter() - started
    with torch.no_grad():
        predicted = trained(test_images.flatten(start_dim=1)).argmax(dim=-1)
        accuracy = (predicted == test_labels).float().mean().item()
    return seconds, steps, accuracy

for batch_size in (1, 32):
    seconds, steps, accuracy = train_small(batch_size)
    print(f"Batch {batch_size:2}: {steps:3} updates; {seconds:.4f}s; accuracy on 32 test images: {accuracy:.1%}")

**Question:** Use batch size 128. Compare updates, elapsed time and accuracy.

## A hidden layer

The first layer produces 128 values. The second produces ten scores.

In [ ]:
torch.manual_seed(17)
two_linear = nn.Sequential(
    nn.Linear(784, 128),
    nn.Linear(128, 10),
)
hidden_values = two_linear[0](x)
stacked_logits = two_linear(x)
print("Hidden shape:", hidden_values.shape)
print("Score shape:", stacked_logits.shape)
print("Parameters:", sum(p.numel() for p in two_linear.parameters()))

**Question:** Change the first layer to 64 outputs. Which dimension in the second layer must change?

## Two linear layers

Combine the weights and biases, then compare the scores.

In [ ]:
first, second = two_linear
collapsed = nn.Linear(784, 10)
with torch.no_grad():
    collapsed.weight.copy_(second.weight @ first.weight)
    collapsed.bias.copy_(second.weight @ first.bias + second.bias)
probe_inputs = test_images.flatten(start_dim=1)
with torch.no_grad():
    stacked_scores = two_linear(probe_inputs)
    single_scores = collapsed(probe_inputs)
print("Same scores:", torch.allclose(stacked_scores, single_scores, atol=1e-6))
print("Largest difference:", (stacked_scores - single_scores).abs().max().item())

**Question:** Insert ReLU between the layers. Do the scores still match the combined layer?

## ReLU

ReLU maps negative inputs to zero and keeps positive inputs.

In [ ]:
values = torch.tensor([-2., -0.5, 0., 0.5, 2.], requires_grad=True)
activation = nn.ReLU()
activated = activation(values)
activated.sum().backward()
print("Input:", values.detach())
print("Output:", activated.detach())
print("Gradient:", values.grad)

**Question:** Replace ReLU with Identity. What outputs and gradients do you get?

## The classifier with ReLU

Apply ReLU to the hidden values before calculating the ten scores.

In [ ]:
class HiddenClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        features = self.activation(self.hidden(x))
        return self.output(features)

torch.manual_seed(17)
hidden_model = HiddenClassifier()
print("One image:", hidden_model(x).shape)
print("32 images:", hidden_model(test_images.flatten(start_dim=1)).shape)
print("Parameters:", sum(p.numel() for p in hidden_model.parameters()))
print("First three scores:", hidden_model(x).detach()[:3])

**Question:** Use Identity in place of ReLU. Which properties of the model change?

## Model comparison

Train each model using the same data, batch size and learning rate.

In [ ]:
class TwoLinearClassifier(HiddenClassifier):
    def __init__(self):
        super().__init__()
        self.activation = nn.Identity()

for name, model_class in [
    ("one linear", DigitClassifier),
    ("two linear", TwoLinearClassifier),
    ("linear + ReLU + linear", HiddenClassifier),
]:
    seconds, steps, accuracy = train_small(32, epochs=20, model_class=model_class)
    print(f"{name}: {steps} updates; {seconds:.4f}s; accuracy on 32 test images: {accuracy:.1%}")

**Question:** Train for 100 epochs. Compare the three results.